## Section I: Prerequisites

### 1.0. Import Required Libraries

In [1]:
import findspark
findspark.init()
print(findspark.find())

import os
import sys
import json
import time
import pymongo
import certifi
import shutil
import pandas as pd

import sqlalchemy
from sqlalchemy import create_engine, text

import pyspark
from pyspark import SparkConf
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W

C:\spark-4.0.2-bin-hadoop3


In [2]:
print(f"SQLAlchemy version : {sqlalchemy.__version__}")
print(f"PyMongo version    : {pymongo.__version__}")
print(f"PySpark version    : {pyspark.__version__}")

SQLAlchemy version : 2.0.34
PyMongo version    : 4.16.0
PySpark version    : 4.0.2


### 2.0. Instantiate Global Variables

In [3]:
# --------------------------------------------------------------------------------
# Specify MySQL Server Connection Information
# --------------------------------------------------------------------------------
mysql_args = {
    "uid" : "lawsonpham",
    "pwd" : "Lawson2006",
    "host_name" : "127.0.0.1",
    "port" : "3306",
    "src_dbname" : "adventureworks",
    "dst_dbname" : "adventureworks_dw2",
    "conn_props" : {
        "user" : "lawsonpham",
        "password" : "Lawson2006",
        "driver" : "com.mysql.cj.jdbc.Driver"
    }
}

# --------------------------------------------------------------------------------
# Specify MongoDB Cluster Connection Information
# --------------------------------------------------------------------------------
mongodb_args = {
    "cluster_location" : "atlas", # "atlas"
    "user_name" : "lawsonpham",
    "password" : "Lawson2006",
    "cluster_name" : "cluster0",
    "cluster_subnet" : "m6yd7zf",
    "db_name" : "adventureworks_dw2",
    "collection" : "",
    "null_column_threshold" : 0.5
}

# --------------------------------------------------------------------------------
# Specify Directory Structure for Source Data
# --------------------------------------------------------------------------------
base_dir = os.path.join(os.getcwd(), 'data')
data_dir = os.path.join(base_dir, 'adventureworks')
batch_dir = os.path.join(data_dir, 'batch')
stream_dir = os.path.join(data_dir, 'streaming')

sales_stream_dir = os.path.join(stream_dir, 'fact_sale_orders')
customer_csv = os.path.join(batch_dir, 'dim_customers.csv')

# --------------------------------------------------------------------------------
# Create Directory Structure for Data Lakehouse Files
# --------------------------------------------------------------------------------
dest_database = "adventureworks_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

sales_output_bronze = os.path.join(database_dir, 'fact_sales_orders', 'bronze')
sales_output_silver = os.path.join(database_dir, 'fact_sales_orders', 'silver')
sales_output_gold = os.path.join(database_dir, 'fact_sales_orders', 'gold')

print("Directory paths configured:")
print(f"  base_dir         : {base_dir}")
print(f"  batch_dir        : {batch_dir}")
print(f"  stream_dir       : {stream_dir}")
print(f"  sales_stream_dir : {sales_stream_dir}")
print(f"  Warehouse        : {sql_warehouse_dir}")
print(f"  Database         : {database_dir}")

Directory paths configured:
  base_dir         : C:\Users\phaml\OneDrive\Documents\GitHub\DS-2002\Capstone\data
  batch_dir        : C:\Users\phaml\OneDrive\Documents\GitHub\DS-2002\Capstone\data\adventureworks\batch
  stream_dir       : C:\Users\phaml\OneDrive\Documents\GitHub\DS-2002\Capstone\data\adventureworks\streaming
  sales_stream_dir : C:\Users\phaml\OneDrive\Documents\GitHub\DS-2002\Capstone\data\adventureworks\streaming\fact_sale_orders
  Warehouse        : C:\Users\phaml\OneDrive\Documents\GitHub\DS-2002\Capstone\spark-warehouse
  Database         : C:\Users\phaml\OneDrive\Documents\GitHub\DS-2002\Capstone\spark-warehouse\adventureworks_dlh.db


### 3.0. Define Global Functions

In [4]:
def get_file_info(path: str):
    file_sizes = []
    modification_times = []

    '''Fetch each item in the directory, and filter out any directories.'''
    items = os.listdir(path)
    files = sorted([item for item in items if os.path.isfile(os.path.join(path, item))])

    '''Populate lists with the Size and Last Modification DateTime for each file in the directory.'''
    for file in files:
        file_sizes.append(os.path.getsize(os.path.join(path, file)))
        modification_times.append(pd.to_datetime(os.path.getmtime(os.path.join(path, file)), unit='s'))

    data = list(zip(files, file_sizes, modification_times))
    column_names = ['name','size','modification_time']
    
    return pd.DataFrame(data=data, columns=column_names)


def wait_until_stream_is_ready(query, min_batches=1):
    while len(query.recentProgress) < min_batches:
        time.sleep(5)
        
    print(f"The stream has processed {len(query.recentProgress)} batchs")


def remove_directory_tree(path: str):
    '''If it exists, remove the entire contents of a directory structure at a given 'path' parameter's location.'''
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
            
    except Exception as e:
        return f"An error occurred: {e}"
        

def drop_null_columns(df, threshold):
    '''Drop Columns having a percentage of NULL values that exceeds the given 'threshold' parameter value.'''
    columns_with_nulls = [col for col in df.columns if df.filter(df[col].isNull()).count() / df.count() > threshold] 
    df_dropped = df.drop(*columns_with_nulls) 
    
    return df_dropped
    
    
def get_mysql_dataframe(spark_session, sql_query : str, **args):
    '''Create a JDBC URL to the MySQL Database'''
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['src_dbname']}"
    
    '''Invoke the spark.read.format("jdbc") function to query the database, and fill a DataFrame.'''
    dframe = spark_session.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("driver", args['conn_props']['driver']) \
    .option("user", args['conn_props']['user']) \
    .option("password", args['conn_props']['password']) \
    .option("query", sql_query) \
    .load()
    
    return dframe
    

def get_mongo_uri(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the 'cluster_location' parameter.")
        
    if args['cluster_location'] == "atlas":
        uri = f"mongodb+srv://{args['user_name']}:{args['password']}@"
        uri += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net/"
    else:
        uri = "mongodb://localhost:27017/"

    return uri


def get_spark_conf_args(spark_jars : list, **args):
    jars = ""
    for jar in spark_jars:
        jars += f"{jar}, "
    
    sparkConf_args = {
        "app_name" : "PySpark Northwind Data Lakehouse (Medallion Architecture)",
        "worker_threads" : f"local[{int(os.cpu_count()/2)}]",
        "shuffle_partitions" : int(os.cpu_count()),
        "mongo_uri" : get_mongo_uri(**args),
        "spark_jars" : jars[0:-2],
        "database_dir" : sql_warehouse_dir
    }
    
    return sparkConf_args
    

def get_spark_conf(**args):
    sparkConf = SparkConf().setAppName(args['app_name'])\
    .setMaster(args['worker_threads']) \
    .set('spark.driver.memory', '4g') \
    .set('spark.executor.memory', '2g') \
    .set('spark.jars', args['spark_jars']) \
    .set('spark.jars.packages', 'org.mongodb.spark:mongo-spark-connector_2.13:10.3.0') \
    .set('spark.mongodb.read.connection.uri', args['mongo_uri']) \
    .set('spark.mongodb.write.connection.uri', args['mongo_uri']) \
    .set('spark.sql.adaptive.enabled', 'false') \
    .set('spark.sql.debug.maxToStringFields', 35) \
    .set('spark.sql.shuffle.partitions', args['shuffle_partitions']) \
    .set('spark.sql.streaming.forceDeleteTempCheckpointLocation', 'true') \
    .set('spark.sql.streaming.schemaInference', 'true') \
    .set('spark.sql.warehouse.dir', args['database_dir']) \
    .set('spark.streaming.stopGracefullyOnShutdown', 'true')
    
    return sparkConf


def get_mongo_client(**args):
    '''Get MongoDB Client Connection'''
    mongo_uri = get_mongo_uri(**args)
    if args['cluster_location'] == "atlas":
        client = pymongo.MongoClient(mongo_uri, tlsCAFile=certifi.where())

    elif args['cluster_location'] == "local":
        client = pymongo.MongoClient(mongo_uri)
        
    else:
        raise Exception("A MongoDB Client could not be created.")

    return client
    
    
# TODO: Rewrite this to leverage PySpark?
def set_mongo_collections(mongo_client, db_name : str, data_directory : str, json_files : list):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()
    

def get_mongodb_dataframe(spark_session, **args):
    """
    Query MongoDB Atlas or local MongoDB and return a Spark DataFrame.
    Drops '_id' and mostly-null columns.
    """
    mongo_uri = get_mongo_uri(**args)

    dframe = spark_session.read.format("mongodb") \
        .option("connection.uri", mongo_uri) \
        .option("database", args['db_name']) \
        .option("collection", args['collection']) \
        .load()

    # Drop '_id' column if it exists
    if "_id" in dframe.columns:
        dframe = dframe.drop("_id")

    # Drop mostly-null columns
    dframe = drop_null_columns(dframe, args['null_column_threshold'])

    return dframe

def get_sql_dataframe(sql_query, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['host_name']}/{args['src_dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    
    '''Invoke the pd.read_sql() function to query the database, and fill a Pandas DataFrame.'''
    dframe = pd.read_sql(text(sql_query), connection);
    connection.close()
    
    return dframe
    

def set_dataframe(df, table_name, pk_column, db_operation, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['host_name']}/{args['dst_dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()
    
    '''Invoke the Pandas DataFrame .to_sql( ) function to either create, or append to, a table'''
    if db_operation in ['insert', 'update']:
        if db_operation.lower() == "insert":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
                    
        elif db_operation.lower() == "update":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append')

    else:
        print("The value supplied to the 'db_operation' parameter must be either 'insert' or 'update'.")
    
    db_connection.close()



### 4.0. Initialize Data Lakehouse Directory Structure
Remove the Data Lakehouse Database Directory Structure to Ensure Idempotency

In [5]:
remove_directory_tree(database_dir)

"Directory 'C:\\Users\\phaml\\OneDrive\\Documents\\GitHub\\DS-2002\\Capstone\\spark-warehouse\\adventureworks_dlh.db' has been removed successfully."

### 5.0. Create a New Spark Session

In [6]:
worker_threads = f"local[{int(os.cpu_count()/2)}]"

jars = []
mysql_spark_jar = os.path.join(os.getcwd(), "mysql-connector-j-9.1.0", "mysql-connector-j-9.1.0.jar")
mssql_spark_jar = os.path.join(os.getcwd(), "sqljdbc_12.8", "enu", "jars", "mssql-jdbc-12.8.1.jre11.jar")

jars.append(mysql_spark_jar)
#jars.append(mssql_spark_jar)

sparkConf_args = get_spark_conf_args(jars, **mongodb_args)

sparkConf = get_spark_conf(**sparkConf_args)
spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
spark.sparkContext.setLogLevel("OFF")
spark

### 6.0. Create a New Metadata Database.

In [7]:
spark.sql(f"DROP DATABASE IF EXISTS {dest_database} CASCADE;")

sql_create_db = f"""
    CREATE DATABASE IF NOT EXISTS {dest_database}
    COMMENT 'DS-2002 Lab 06 Database'
    WITH DBPROPERTIES (contains_pii = true, purpose = 'DS-2002 Capstone');
"""
spark.sql(sql_create_db)

DataFrame[]

## Section II: Populate Dimensions by Ingesting "Cold-path" Reference Data 
### 1.0. Fetch Data from the File System
#### 1.1. Verify the location of the source data files on the file system

In [8]:
get_file_info(batch_dir)

,name,size,modification_time
0,dim_customers.csv,130435,2026-03-22 22:38:42.394726992
1,dim_employees.json,161452,2026-03-22 22:40:22.195275784


#### 1.2. Populate the <span style="color:darkred">Customers Dimension</span> (Batch Execution)
##### 1.2.1. Use PySpark to Read data from a CSV file

In [9]:
customer_csv = os.path.join(batch_dir, 'dim_customers.csv')
print(customer_csv)

df_dim_customers = spark.read.format('csv').options(header='true', inferSchema='true').load(customer_csv)
df_dim_customers.toPandas().head(2)

C:\Users\phaml\OneDrive\Documents\GitHub\DS-2002\Capstone\data\adventureworks\batch\dim_customers.csv


,CustomerID,AccountNumber,CustomerType,AddressType,AddressLine1,AddressLine2,City,StateProvinceCode,State_Province,IsOnlyStateProvinceFlag,PostalCode,CountryRegionCode,Country_Region,Sales Territory Group,Sales Territory
0,1,AW00000001,S,Main Office,2251 Elliot Avenue,NULL,Seattle,WA,Washington,0,98104,US,United States,North America,Northwest
1,2,AW00000002,S,Shipping,7943 Walnut Ave,NULL,Renton,WA,Washington,0,98055,US,United States,North America,Northwest


##### 1.2.2. Make Necessary Transformations to the New DataFrame

In [10]:
# ----------------------------------------------------------------------------------
# Add Primary Key column using the SQL Windowing function: ROW_NUMBER() 
# ----------------------------------------------------------------------------------
df_dim_customers.createOrReplaceTempView("customers")
sql_customers = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY CustomerID) AS customer_key
    FROM customers;
"""
df_dim_customers = spark.sql(sql_customers)

# ----------------------------------------------------------------------------------
# Reorder Columns and display the first two rows in a Pandas dataframe
# ----------------------------------------------------------------------------------
ordered_columns = ['customer_key', 'CustomerID', 'AccountNumber', 'CustomerType'
                   , 'AddressType', 'AddressLine1', 'City', 'StateProvinceCode'
                   , 'State_Province', 'PostalCode', 'CountryRegionCode'
                   , 'Country_Region', 'Sales Territory Group', 'Sales Territory']
df_dim_customers = df_dim_customers[ordered_columns]
df_dim_customers.toPandas().head(2)

,customer_key,CustomerID,AccountNumber,CustomerType,AddressType,AddressLine1,City,StateProvinceCode,State_Province,PostalCode,CountryRegionCode,Country_Region,Sales Territory Group,Sales Territory
0,1,1,AW00000001,S,Main Office,2251 Elliot Avenue,Seattle,WA,Washington,98104,US,United States,North America,Northwest
1,2,2,AW00000002,S,Shipping,7943 Walnut Ave,Renton,WA,Washington,98055,US,United States,North America,Northwest


##### 1.2.3. Save as the <span style="color:darkred">dim_customers</span> table in the Data Lakehouse

In [11]:
df_dim_customers.write.saveAsTable(f"{dest_database}.dim_customers", mode="overwrite")

##### 1.2.4. Unit Test: Describe and Preview Table

In [12]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_customers;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_customers LIMIT 2").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|        customer_key|                 int|   NULL|
|          CustomerID|                 int|   NULL|
|       AccountNumber|              string|   NULL|
|        CustomerType|              string|   NULL|
|         AddressType|              string|   NULL|
|        AddressLine1|              string|   NULL|
|                City|              string|   NULL|
|   StateProvinceCode|              string|   NULL|
|      State_Province|              string|   NULL|
|          PostalCode|              string|   NULL|
|   CountryRegionCode|              string|   NULL|
|      Country_Region|              string|   NULL|
|Sales Territory G...|              string|   NULL|
|     Sales Territory|              string|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|           

,customer_key,CustomerID,AccountNumber,CustomerType,AddressType,AddressLine1,City,StateProvinceCode,State_Province,PostalCode,CountryRegionCode,Country_Region,Sales Territory Group,Sales Territory
0,1,1,AW00000001,S,Main Office,2251 Elliot Avenue,Seattle,WA,Washington,98104,US,United States,North America,Northwest
1,2,2,AW00000002,S,Shipping,7943 Walnut Ave,Renton,WA,Washington,98055,US,United States,North America,Northwest


### 2.0. Fetch Reference Data from a MySQL Database
#### 2.1. Populate the <span style="color:darkred">Date Dimension</span>
##### 2.1.1 Fetch data from the <span style="color:darkred">dim_date</span> table in MySQL

In [13]:
sql_dim_date = f"SELECT * FROM {mysql_args['src_dbname']}.dim_date"
df_dim_date = get_mysql_dataframe(spark, sql_dim_date, **mysql_args)

##### 2.1.2. Save as the <span style="color:darkred">dim_date</span> table in the Data Lakehouse

In [14]:
df_dim_date.write.saveAsTable(f"{dest_database}.dim_date", mode="overwrite")

##### 2.1.3. Unit Test: Describe and Preview Table

In [15]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_date;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_date LIMIT 2").toPandas()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|            date_key|      int|   NULL|
|           full_date|     date|   NULL|
|           date_name| char(11)|   NULL|
|        date_name_us| char(11)|   NULL|
|        date_name_eu| char(11)|   NULL|
|         day_of_week|  tinyint|   NULL|
|    day_name_of_week| char(10)|   NULL|
|        day_of_month|  tinyint|   NULL|
|         day_of_year| smallint|   NULL|
|     weekday_weekend| char(10)|   NULL|
|        week_of_year|  tinyint|   NULL|
|          month_name| char(10)|   NULL|
|       month_of_year|  tinyint|   NULL|
|is_last_day_of_month|  char(1)|   NULL|
|    calendar_quarter|  tinyint|   NULL|
|       calendar_year| smallint|   NULL|
| calendar_year_month| char(10)|   NULL|
|   calendar_year_qtr| char(10)|   NULL|
|fiscal_month_of_year|  tinyint|   NULL|
|      fiscal_quarter|  tinyint|   NULL|
+--------------------+---------+-------+
only showing top

,date_key,full_date,date_name,date_name_us,date_name_eu,day_of_week,day_name_of_week,day_of_month,day_of_year,weekday_weekend,...,is_last_day_of_month,calendar_quarter,calendar_year,calendar_year_month,calendar_year_qtr,fiscal_month_of_year,fiscal_quarter,fiscal_year,fiscal_year_month,fiscal_year_qtr
0,20000101,2000-01-01,2000/01/01,01/01/2000,01/01/2000,7,Saturday,1,1,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3
1,20000102,2000-01-02,2000/01/02,01/02/2000,02/01/2000,1,Sunday,2,2,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3


#### 2.2. Populate the <span style="color:darkred">Product Dimension</span>
##### 2.2.1. Fetch data from the <span style="color:darkred">Products</span> table in MySQL

In [16]:
# ----------------------------------------------------------------------------------
# Add Primary Key column using the SQL Windowing function: ROW_NUMBER() 
# ----------------------------------------------------------------------------------
sql_products = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY ProductID) AS product_key
    FROM {mysql_args['src_dbname']}.dim_products_vw
"""
df_dim_products = get_mysql_dataframe(spark, sql_products, **mysql_args)

In [17]:
# ----------------------------------------------------------------------------------
# Drop unwanted columns (description and attachments)
# ----------------------------------------------------------------------------------
for drop_column in ['description','attachments']:
    df_dim_products = df_dim_products.drop(drop_column)

# ----------------------------------------------------------------------------------
# Reorder Columns and display the first two rows in a Pandas dataframe
# ----------------------------------------------------------------------------------
ordered_columns = ['product_key', 'ProductID', 'Name', 'ProductNumber'
                   , 'Color', 'SafetyStockLevel', 'ReorderPoint', 'StandardCost'
                   , 'ListPrice']
df_dim_products = df_dim_products[ordered_columns]
df_dim_products.toPandas().head(2)

,product_key,ProductID,Name,ProductNumber,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice
0,1,1,Adjustable Race,AR-5381,None,1000,750,0.0,0.0
1,2,2,Bearing Ball,BA-8327,None,1000,750,0.0,0.0


##### 2.2.3. Save as the <span style="color:darkred">dim_products</span> table in the Data Lakehouse

In [18]:
df_dim_products.write.saveAsTable(f"{dest_database}.dim_products", mode="overwrite")

##### 2.2.4. Unit Test: Describe and Preview Table

In [19]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_products;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_products LIMIT 2").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|         product_key|       decimal(20,0)|   NULL|
|           ProductID|                 int|   NULL|
|                Name|         varchar(50)|   NULL|
|       ProductNumber|         varchar(25)|   NULL|
|               Color|         varchar(15)|   NULL|
|    SafetyStockLevel|            smallint|   NULL|
|        ReorderPoint|            smallint|   NULL|
|        StandardCost|              double|   NULL|
|           ListPrice|              double|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|  adventureworks_dlh|       |
|               Table|        dim_products|       |
|        Created Time|Thu Apr 30 22:34:...|       |
|         Last Access|             UNKNOWN|       |
|          C

,product_key,ProductID,Name,ProductNumber,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice
0,1,1,Adjustable Race,AR-5381,None,1000,750,0.0,0.0
1,2,2,Bearing Ball,BA-8327,None,1000,750,0.0,0.0


### 3.0. Fetch Reference Data from a MongoDB Atlas Database
#### 3.1. Create a New MongoDB Database, and Load Each JSON File into a New MongoDB Collection
**NOTE:** The following cell **can** be run more than once because the **set_mongo_collection()** function **is** idempotent.

In [20]:
client = get_mongo_client(**mongodb_args)

json_files = {"employees" : "dim_employees.json"}

set_mongo_collections(client, mongodb_args["db_name"], batch_dir, json_files) 

#### 3.2. Populate the <span style="color:darkred">Employees Dimension</span>
##### 3.2.1. Fetch Data from the New MongoDB <span style="color:darkred">Employees</span> Collection

In [21]:
mongodb_args["collection"] = "employees"

df_dim_employees = get_mongodb_dataframe(spark, **mongodb_args)
df_dim_employees.toPandas().head(2)

,BirthDate,CurrentFlag,EmailAddress,EmailPromotion,EmployeeID,FirstName,Gender,HireDate,LastName,LoginID,ManagerID,MaritalStatus,MiddleName,NationalIDNumber,Phone,SalariedFlag,SickLeaveHours,Title,VacationHours
0,1972-05-15 00:00:00,1,guy1@adventure-works.com,0,1,Guy,M,1996-07-31 00:00:00,Gilbert,adventure-works\guy1,16.0,M,R,14417807,320-555-0195,0,30,Production Technician - WC60,21
1,1977-06-03 00:00:00,1,kevin0@adventure-works.com,2,2,Kevin,M,1997-02-26 00:00:00,Brown,adventure-works\kevin0,6.0,S,F,253022876,150-555-0189,0,41,Marketing Assistant,42


##### 3.2.2. Make Necessary Transformations to the New Dataframe

In [22]:
# ----------------------------------------------------------------------------------
# Add Primary Key column using the SQL Windowing function: ROW_NUMBER() 
# ----------------------------------------------------------------------------------
df_dim_employees.createOrReplaceTempView("employees")
sql_employees = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY EmployeeID) AS employee_key
    FROM employees;
"""
df_dim_employees = spark.sql(sql_employees)

# ----------------------------------------------------------------------------------
# Reorder Columns and display the first two rows in a Pandas dataframe
# ----------------------------------------------------------------------------------
ordered_columns = ['employee_key', 'EmployeeID', 'FirstName', 'MiddleName'
                   , 'LastName', 'Title', 'EmailAddress', 'Phone'
                   , 'BirthDate', 'MaritalStatus', 'Gender', 'HireDate'
                   , 'VacationHours', 'SickLeaveHours']
df_dim_employees = df_dim_employees[ordered_columns]
df_dim_employees.toPandas().head(2)

,employee_key,EmployeeID,FirstName,MiddleName,LastName,Title,EmailAddress,Phone,BirthDate,MaritalStatus,Gender,HireDate,VacationHours,SickLeaveHours
0,1,1,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,320-555-0195,1972-05-15 00:00:00,M,M,1996-07-31 00:00:00,21,30
1,2,2,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,150-555-0189,1977-06-03 00:00:00,S,M,1997-02-26 00:00:00,42,41


##### 3.2.3. Save as the <span style="color:darkred">dim_employees</span> table in the Data lakehouse

In [23]:
df_dim_employees.write.saveAsTable(f"{dest_database}.dim_employees", mode="overwrite")

##### 3.2.4. Unit Test: Describe and Preview Table

In [24]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_employees;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_employees LIMIT 2").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|        employee_key|                 int|   NULL|
|          EmployeeID|                 int|   NULL|
|           FirstName|              string|   NULL|
|          MiddleName|              string|   NULL|
|            LastName|              string|   NULL|
|               Title|              string|   NULL|
|        EmailAddress|              string|   NULL|
|               Phone|              string|   NULL|
|           BirthDate|              string|   NULL|
|       MaritalStatus|              string|   NULL|
|              Gender|              string|   NULL|
|            HireDate|              string|   NULL|
|       VacationHours|                 int|   NULL|
|      SickLeaveHours|                 int|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|           

,employee_key,EmployeeID,FirstName,MiddleName,LastName,Title,EmailAddress,Phone,BirthDate,MaritalStatus,Gender,HireDate,VacationHours,SickLeaveHours
0,1,1,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,320-555-0195,1972-05-15 00:00:00,M,M,1996-07-31 00:00:00,21,30
1,2,2,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,150-555-0189,1977-06-03 00:00:00,S,M,1997-02-26 00:00:00,42,41


### 4.0 Verify Dimension Tables

In [25]:
spark.sql(f"USE {dest_database};")
spark.sql("SHOW TABLES").toPandas()

,namespace,tableName,isTemporary
0,adventureworks_dlh,dim_customers,False
1,adventureworks_dlh,dim_date,False
2,adventureworks_dlh,dim_employees,False
3,adventureworks_dlh,dim_products,False
4,,customers,True
5,,employees,True


## Section III: Integrate Reference Data with Real-Time Data
### 1.0. Use PySpark Structured Streaming to Process (Hot Path) <span style="color:darkred">Sales Order</span> Fact Data  
#### 1.1. Verify the location of the source data files on the file system

In [26]:
get_file_info(sales_stream_dir)

,name,size,modification_time
0,fact_sale_orders_01.json,309109,2026-04-29 23:19:34.494398355
1,fact_sale_orders_02.json,306326,2026-04-29 23:20:03.327081919
2,fact_sale_orders_03.json,310080,2026-04-29 23:20:24.812123299


#### 1.2. Create the Bronze Layer: Stage <span style="color:darkred">Orders Fact table</span> Data
##### 1.2.1. Read "Raw" JSON file data into a Stream

In [27]:
df_sales_bronze = (
    spark.readStream \
    .option("schemaLocation", sales_output_bronze) \
    .option("maxFilesPerTrigger", 1) \
    .option("multiLine", "true") \
    .json(sales_stream_dir)
)

df_sales_bronze.isStreaming

True

##### 1.2.2. Write the Streaming Data to a Parquet file

In [28]:
sales_checkpoint_bronze = os.path.join(sales_output_bronze, '_checkpoint')

sales_bronze_query = (
    df_sales_bronze
    # Add Current Timestamp and Input Filename columns for Traceability
    .withColumn("receipt_time", current_timestamp())
    .withColumn("source_file", input_file_name())
    
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("sales_bronze")
    .trigger(availableNow = True) \
    .option("checkpointLocation", sales_checkpoint_bronze) \
    .option("compression", "snappy") \
    .start(sales_output_bronze)
)

##### 1.2.3. Unit Test: Implement Query Monitoring

In [29]:
print(f"Query ID: {sales_bronze_query.id}")
print(f"Query Name: {sales_bronze_query.name}")
print(f"Query Status: {sales_bronze_query.status}")

Query ID: 46818542-c443-46c9-9620-c2c36f67f4ff
Query Name: sales_bronze
Query Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}


In [30]:
sales_bronze_query.awaitTermination()

#### 1.3. Create the Silver Layer: Integrate "Cold-path" Data & Make Transformations
##### 1.3.1. Prepare Role-Playing Dimension Primary and Business Keys

In [31]:
df_dim_order_date = df_dim_date.select(col("date_key").alias("order_date_key"), col("full_date").alias("order_full_date"))
df_dim_due_date = df_dim_date.select(col("date_key").alias("due_date_key"), col("full_date").alias("due_full_date"))
df_dim_ship_date = df_dim_date.select(col("date_key").alias("ship_date_key"), col("full_date").alias("ship_full_date"))

##### 1.3.2. Define Silver Query to Join Streaming with Batch Data

In [32]:
df_sales_silver = spark.readStream.format("parquet").load(sales_output_bronze) \
    .join(df_dim_customers, "CustomerID") \
    .join(df_dim_employees, col("SalesPersonID") == df_dim_employees.EmployeeID, "inner") \
    .join(df_dim_products, "ProductID") \
    .join(df_dim_order_date, df_dim_order_date.order_full_date.cast(DateType()) == col("OrderDate").cast(DateType()), "inner") \
    .join(df_dim_ship_date, df_dim_ship_date.ship_full_date.cast(DateType()) == col("ShipDate").cast(DateType()), "left_outer") \
    .join(df_dim_due_date, df_dim_due_date.due_full_date.cast(DateType()) == col("DueDate").cast(DateType()), "left_outer") \
    .select(col("SalesOrderID").cast(LongType()), \
            col("SalesPersonID").cast(LongType()), \
            df_dim_customers.customer_key.cast(LongType()), \
            df_dim_employees.employee_key.cast(LongType()), \
            df_dim_products.product_key.cast(LongType()), \
            df_dim_order_date.order_date_key.cast(LongType()), \
            df_dim_due_date.due_date_key.cast(LongType()), \
            df_dim_ship_date.ship_date_key.cast(LongType()), \
            col("Status"), \
            col("OrderQty"), \
            col("UnitPrice"), \
            col("LineTotal"), \
            col("ShipMethod"), \
            col("ShipBase"), \
            col("ShipRate"), \
            col("Credit Card Type"), \
            col("SubTotal"), \
            col("TaxAmt"), \
            col("Freight"), \
            col("TotalDue"), \
            col("receipt_time"), \
            col("source_file") \
           )

In [33]:
df_sales_silver.isStreaming

True

In [34]:
df_sales_silver.printSchema()

root
 |-- SalesOrderID: long (nullable = true)
 |-- SalesPersonID: long (nullable = true)
 |-- customer_key: long (nullable = false)
 |-- employee_key: long (nullable = false)
 |-- product_key: long (nullable = true)
 |-- order_date_key: long (nullable = true)
 |-- due_date_key: long (nullable = true)
 |-- ship_date_key: long (nullable = true)
 |-- Status: long (nullable = true)
 |-- OrderQty: long (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- LineTotal: double (nullable = true)
 |-- ShipMethod: string (nullable = true)
 |-- ShipBase: double (nullable = true)
 |-- ShipRate: double (nullable = true)
 |-- Credit Card Type: string (nullable = true)
 |-- SubTotal: double (nullable = true)
 |-- TaxAmt: double (nullable = true)
 |-- Freight: double (nullable = true)
 |-- TotalDue: double (nullable = true)
 |-- receipt_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



##### 1.3.3. Write the Transformed Streaming data to the Data Lakehouse

In [35]:
sales_checkpoint_silver = os.path.join(sales_output_silver, '_checkpoint')

sales_silver_query = (
    df_sales_silver.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("sales_silver")
    .trigger(availableNow = True) \
    .option("checkpointLocation", sales_checkpoint_silver) \
    .option("compression", "snappy") \
    .start(sales_output_silver)
)

##### 1.3.4. Unit Test: Implement Query Monitoring

In [36]:
print(f"Query ID: {sales_silver_query.id}")
print(f"Query Name: {sales_silver_query.name}")
print(f"Query Status: {sales_silver_query.status}")

Query ID: 6d8d3525-ed36-437f-b119-eca89dc9d4a4
Query Name: sales_silver
Query Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}


In [37]:
sales_silver_query.awaitTermination()

#### 1.4. Create Gold Layer: Perform Aggregations
##### 1.4.1. Define a Query to Create a Business Report

In [38]:
df_sales_gold = spark.readStream.format("parquet").load(sales_output_silver) \
.join(df_dim_customers, "customer_key") \
.join(df_dim_products, "product_key") \
.join(df_dim_date, df_dim_date.date_key.cast(IntegerType()) == col("order_date_key").cast(IntegerType())) \
.groupBy(df_dim_customers.CustomerID, df_dim_customers.City,"Country_Region", "calendar_year", "month_name") \
.agg(count("SalesOrderID").alias("total_orders"), sum("LineTotal").alias("total_revenue")) \
.orderBy(desc("total_revenue"))

In [39]:
df_sales_gold.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- Country_Region: string (nullable = true)
 |-- calendar_year: short (nullable = true)
 |-- month_name: string (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_revenue: double (nullable = true)



##### 1.4.2. Write the Streaming data to a Parquet File in "Complete" mode

In [40]:
 sales_gold_query = (
    df_sales_gold.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("sales_gold")
    .start()
)

In [57]:
wait_until_stream_is_ready(sales_gold_query, 1)

The stream has processed 13 batchs


##### 1.4.3. Query the Gold Data from Memory

In [58]:
df_sales_gold = spark.sql("SELECT * FROM sales_gold")
df_sales_gold.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- Country_Region: string (nullable = true)
 |-- calendar_year: short (nullable = true)
 |-- month_name: string (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_revenue: double (nullable = true)



##### 1.4.4 Create the Final Selection

In [59]:
df_sales_gold_final = df_sales_gold \
.select(col("CustomerID").alias("Customer ID"), \
        col("City").alias("City"), \
        col("Country_Region").alias("Country"), \
        col("calendar_year").alias("Year"), \
        col("month_name").alias("Month"), \
        col("total_orders").alias("Total Orders"), \
        col("total_revenue").alias("Total Revenue")) \
.orderBy(desc("Total Revenue"))

##### 1.4.5. Load the Final Results into a New Table and Display the Results

In [60]:
df_sales_gold_final.write.saveAsTable(f"{dest_database}.sales_gold", mode="overwrite")
spark.sql(f"SELECT * FROM {dest_database}.sales_gold").toPandas()

,Customer ID,City,Country,Year,Month,Total Orders,Total Revenue
0,578,Puyallup,United States,2001,July,1,714.7043
1,22,North Sioux City,United States,2001,August,1,419.4589
2,510,Trabuco Canyon,United States,2001,July,1,419.4589
3,603,Miami,United States,2001,July,1,419.4589
4,623,Minneapolis,United States,2001,August,2,57.6808
...,...,...,...,...,...,...,...
75,166,Garland,United States,2001,July,21,33997.3702
76,208,Toronto,Canada,2001,August,1,1749.5880
77,471,San Antonio,United States,2001,August,4,1718.8983
78,480,Mississauga,Canada,2001,July,3,1316.0575


## Section IV: Business Value Query

In [61]:
sql_business_query = f"""
    SELECT
        `Customer ID`,
        `City`,
        `Country`,
        `Year`,
        `Month`,
        `Total Orders`,
        `Total Revenue`
    FROM {dest_database}.sales_gold
    ORDER BY `Total Revenue` DESC
    LIMIT 10
"""

df_result = spark.sql(sql_business_query)
print("Top 10 Customers by Revenue:")
df_result.toPandas()

Top 10 Customers by Revenue:


,Customer ID,City,Country,Year,Month,Total Orders,Total Revenue
0,278,Orlando,United States,2001,August,42,243523.87920
1,650,Reno,United States,2001,August,10,49193.98640
2,506,Casper,United States,2001,July,13,42813.43330
3,78,Ontario,United States,2001,August,28,41250.43910
4,27,Millington,United States,2001,July,8,39373.78100
5,346,Moline,United States,2001,August,8,38816.80555
6,221,Las Cruces,United States,2001,July,28,38510.89730
7,384,San Ysidro,United States,2001,August,9,38474.35720
8,514,Richmond,Canada,2001,July,29,35944.15620
9,166,Garland,United States,2001,July,21,33997.37020


In [62]:
sql_country_year = f"""
    SELECT
        `Country`,
        `Year`,
        SUM(`Total Orders`) AS `Total Orders Sum`,
        SUM(`Total Revenue`) AS `Total Revenue Sum`
    FROM {dest_database}.sales_gold
    GROUP BY `Country`, `Year`
    ORDER BY `Total Revenue Sum` DESC
"""

print("Revenue by Country and Year:")
spark.sql(sql_country_year).toPandas()

Revenue by Country and Year:


,Country,Year,Total Orders Sum,Total Revenue Sum
0,United States,2001,591,1.082953e+06
1,Canada,2001,185,2.393390e+05


### Stop the Spark Session

In [63]:
spark.stop()